<a href="https://colab.research.google.com/github/mdaminu2002-sketch/bank_fraud/blob/main/waste_from_scrach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("joebeachcapital/realwaste")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'realwaste' dataset.
Path to dataset files: /kaggle/input/realwaste


In [6]:
print(os.listdir(path))

['realwaste-main']


In [7]:
for item in os.listdir(path):
  full_path = os.path.join(path, item)
  if os.path.isfile(full_path):
    print(item)
  else:
    print(f"{item} is a folder")

realwaste-main is a folder


In [8]:
inner_path = os.path.join(path, 'realwaste-main')
print(os.listdir(inner_path))

['README.md', 'RealWaste']


In [9]:
for item in os.listdir(inner_path):
  full_path = os.path.join(inner_path, item)
  if os.path.isfile(full_path):
    print(item)
  else:
    print(f"{item} is a folder")

README.md
RealWaste is a folder


In [10]:
data_path = os.path.join(inner_path, 'RealWaste')
print(os.listdir(data_path))

['Metal', 'Glass', 'Paper', 'Vegetation', 'Cardboard', 'Textile Trash', 'Food Organics', 'Plastic', 'Miscellaneous Trash']


In [11]:
for item in os.listdir(data_path):
  full_path = os.path.join(data_path, item)
  if os.path.isdir(full_path):
    count = len(os.listdir(full_path))
    print(f"{item} has {count} images")

Metal has 790 images
Glass has 420 images
Paper has 500 images
Vegetation has 436 images
Cardboard has 461 images
Textile Trash has 318 images
Food Organics has 411 images
Plastic has 921 images
Miscellaneous Trash has 495 images


In [12]:
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

In [13]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [14]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        images, labels = self.subset[index]
        images = self.transform(images)
        return images, labels

In [15]:

full_set = ImageFolder(data_path)
print(full_set)
train_size = int(0.7 * len(full_set))
val_size = int(0.15 * len(full_set))
test_size = len(full_set) - train_size - val_size

train_split, val_split, test_split = random_split(full_set, [train_size, val_size, test_size],
                                                 generator=torch.Generator().manual_seed(42))





train_set = TransformSubset(train_split, train_transform)
val_set = TransformSubset(val_split, val_test_transform)
test_set = TransformSubset(test_split, val_test_transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

Dataset ImageFolder
    Number of datapoints: 4752
    Root location: /kaggle/input/realwaste/realwaste-main/RealWaste


In [16]:
class WasteCNN(nn.Module):
    def __init__(self, num_classes=9):
        super(WasteCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 224 → 112

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 112 → 56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 56 → 28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_scratch = WasteCNN(num_classes=9).to(device)
print(f"Total parameters: {sum(p.numel() for p in model_scratch.parameters())}")

Total parameters: 25786377


In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    # --- TRAIN ---
    model_scratch.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model_scratch(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total

    # --- VAL ---
    model_scratch.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_scratch(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/3 | train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f} | train acc: {train_acc*100:.2f}% | val acc: {val_acc*100:.2f}%")

print("Done!")

Epoch 1/3 | train loss: 2.0368 | val loss: 1.9426 | train acc: 26.34% | val acc: 30.34%
Epoch 2/3 | train loss: 1.9567 | val loss: 1.8255 | train acc: 27.54% | val acc: 30.48%
Epoch 3/3 | train loss: 1.8888 | val loss: 1.7365 | train acc: 28.74% | val acc: 38.76%
Done!
